# Task 3 — Segmentation (§2.3 Transfer Learning · §2.6 Evaluation · §2.7 Pipeline · §2.8 Meta)

**Kaggle setup:** Accelerator → GPU T4x2 · Internet ON

**Prerequisites:** Task 1 done (need `CLASSIFIER_DRIVE_ID`).

**After this notebook you will have:**
- `checkpoints/unet.pth` saved to Kaggle output
- W&B runs for all 3 freeze modes (§2.3) + sample masks (§2.6) + pipeline table (§2.7)
- `UNET_DRIVE_ID` updated in `models/multitask.py` and pushed to GitHub
- Ready to submit **4.3a / 4.3b / 4.3c** to Gradescope

## 1 · Setup

In [ ]:
import os

os.environ["WANDB_API_KEY"] = "PASTE_YOUR_WANDB_API_KEY_HERE"

GITHUB_TOKEN    = "PASTE_YOUR_GITHUB_TOKEN_HERE"
GITHUB_USERNAME = "usnaveen"
GITHUB_REPO     = "A2_Deep_Learning"

!git clone https://{GITHUB_TOKEN}@github.com/{GITHUB_USERNAME}/{GITHUB_REPO}.git da6401_assignment_2
%cd da6401_assignment_2

!pip install -q wandb albumentations gdown scikit-learn

import torch
print(f"CUDA available: {torch.cuda.is_available()}")

## 2 · Download Dataset

In [ ]:
from data.pets_dataset import download_oxford_pet
download_oxford_pet("./data/oxford_pet")

## 3 · Download `classifier.pth` from Google Drive

The segmentation training loads the pretrained VGG11 encoder from the classifier
checkpoint.  This is required for the transfer-learning freeze comparison.

In [ ]:
# ── Paste the CLASSIFIER_DRIVE_ID from Task 1 ─────────────────────────────────
CLASSIFIER_DRIVE_ID = "PASTE_CLASSIFIER_DRIVE_ID_HERE"
# ─────────────────────────────────────────────────────────────────────────────

import gdown, os
os.makedirs("checkpoints", exist_ok=True)
gdown.download(id=CLASSIFIER_DRIVE_ID, output="checkpoints/classifier.pth", quiet=False, fuzzy=True)
print(f"✅  classifier.pth downloaded ({os.path.getsize('checkpoints/classifier.pth')/1e6:.1f} MB)")

## 4 · §2.3 — Transfer Learning Showdown

Three runs with the same U-Net but different encoder freeze strategies:
- **frozen**: entire encoder frozen → only decoder trains
- **partial**: early blocks (0-2) frozen → late blocks (3-4) + decoder train
- **full**: all parameters trainable (full fine-tuning)

All three are logged to the same W&B experiment for side-by-side comparison.

In [ ]:
# Frozen encoder — fastest but lowest ceiling
!python train.py --task segment --experiment exp-transfer --freeze-mode frozen  --epochs 60 --lr 1e-3 --batch-size 16 --num-workers 2

In [ ]:
# Partial freeze
!python train.py --task segment --experiment exp-transfer --freeze-mode partial --epochs 60 --lr 5e-4 --batch-size 16 --num-workers 2

In [ ]:
# Full fine-tuning — best model, save this one for submission
!python train.py --task segment --experiment exp-transfer --freeze-mode full    --epochs 80 --lr 3e-4 --batch-size 16 --num-workers 2

## 5 · Check Best Checkpoint Dice Score

In [ ]:
import torch, numpy as np
from data.pets_dataset import create_dataloaders
from models.segmentation import VGG11UNet

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
_, _, test_loader = create_dataloaders(root="./data/oxford_pet", batch_size=16, num_workers=2)

model = VGG11UNet(num_classes=3).to(device)
model.load_state_dict(torch.load("checkpoints/unet.pth", map_location=device, weights_only=False))
model.eval()

def dice(pred, target, nc=3, eps=1e-6):
    scores = []
    pf, tf = pred.view(-1), target.view(-1)
    for c in range(nc):
        pc = (pf==c).float(); tc = (tf==c).float()
        scores.append((2*(pc*tc).sum()+eps)/(pc.sum()+tc.sum()+eps))
    return np.mean([s.item() for s in scores])

total_dice, n = 0.0, 0
with torch.no_grad():
    for batch in test_loader:
        logits = model(batch["image"].to(device))
        preds  = logits.argmax(1)
        masks  = batch["mask"].to(device)
        total_dice += dice(preds, masks) * len(preds)
        n += len(preds)

mean_dice = total_dice / n
print(f"\n✅  Test Macro-Dice = {mean_dice:.4f}")
print("   (need > 0.30 for 4.3a · > 0.50 for 4.3b · > 0.80 for 4.3c)")

## 6 · §2.7 — Final Pipeline Showcase

Instantiate `MultiTaskPerceptionModel` (shared backbone) and log a W&B table
showing all 3 predictions simultaneously on test images.

In [ ]:
import wandb, torch, numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as patches
from io import BytesIO
from data.pets_dataset import create_dataloaders, IMAGENET_MEAN, IMAGENET_STD, OxfordIIITPetDataset
from models.multitask import MultiTaskPerceptionModel

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
_, _, test_loader = create_dataloaders(root="./data/oxford_pet", batch_size=8, num_workers=2)

# Load the multi-task model from LOCAL checkpoints (skip Drive download)
model = MultiTaskPerceptionModel(
    classifier_path="checkpoints/classifier.pth",
    localizer_path="checkpoints/localizer.pth",
    unet_path="checkpoints/unet.pth",
).to(device)
model.eval()

CLASS_NAMES = OxfordIIITPetDataset.CLASS_NAMES
COLORMAP = np.array([[0,200,0],[40,40,40],[255,255,0]], dtype=np.uint8)

def denorm(t):
    mean = torch.tensor(IMAGENET_MEAN).view(3,1,1)
    std  = torch.tensor(IMAGENET_STD).view(3,1,1)
    img  = (t.cpu()*std+mean).clamp(0,1).permute(1,2,0).numpy()
    return (img*255).astype(np.uint8)

run = wandb.init(project="da6401-assignment2", name="pipeline_showcase", tags=["exp-pipeline"])
table = wandb.Table(columns=["Image", "GT Class", "Pred Class", "IoU", "GT Mask", "Pred Mask"])

count = 0
with torch.no_grad():
    for batch in test_loader:
        imgs   = batch["image"].to(device)
        out    = model(imgs)
        cls_logits = out["classification"].cpu()
        pred_boxes = out["localization"].cpu()
        seg_logits = out["segmentation"].cpu()

        gt_boxes  = batch["bbox"]
        gt_labels = batch["label"]
        gt_masks  = batch["mask"]
        pred_cls  = cls_logits.argmax(1)
        pred_masks= seg_logits.argmax(1)

        for i in range(imgs.size(0)):
            if count >= 20:
                break
            img_np = denorm(imgs[i])
            gt  = gt_boxes[i].numpy()
            pred= pred_boxes[i].numpy()

            # IoU
            px1,py1 = pred[0]-pred[2]/2, pred[1]-pred[3]/2
            px2,py2 = pred[0]+pred[2]/2, pred[1]+pred[3]/2
            tx1,ty1 = gt[0]-gt[2]/2,     gt[1]-gt[3]/2
            tx2,ty2 = gt[0]+gt[2]/2,     gt[1]+gt[3]/2
            ix1,iy1,ix2,iy2 = max(px1,tx1),max(py1,ty1),min(px2,tx2),min(py2,ty2)
            inter = max(0,ix2-ix1)*max(0,iy2-iy1)
            union = (px2-px1)*(py2-py1)+(tx2-tx1)*(ty2-ty1)-inter
            iou_val = inter/(union+1e-6)

            # Draw bbox overlay
            fig, ax = plt.subplots(figsize=(4,4))
            ax.imshow(img_np)
            ax.add_patch(patches.Rectangle((gt[0]-gt[2]/2, gt[1]-gt[3]/2), gt[2], gt[3],
                         linewidth=2, edgecolor='green', facecolor='none', label='GT'))
            ax.add_patch(patches.Rectangle((pred[0]-pred[2]/2, pred[1]-pred[3]/2), pred[2], pred[3],
                         linewidth=2, edgecolor='red', facecolor='none', label='Pred'))
            ax.set_title(f"IoU={iou_val:.2f}"); ax.legend(fontsize=7); ax.axis("off")
            plt.tight_layout()
            buf = BytesIO(); fig.savefig(buf, format='png', dpi=80, bbox_inches='tight'); buf.seek(0); plt.close(fig)

            gt_colored   = COLORMAP[gt_masks[i].numpy()]
            pred_colored = COLORMAP[pred_masks[i].numpy()]

            table.add_data(
                wandb.Image(buf),
                CLASS_NAMES[gt_labels[i].item()] if gt_labels[i].item() < len(CLASS_NAMES) else str(gt_labels[i].item()),
                CLASS_NAMES[pred_cls[i].item()]  if pred_cls[i].item()  < len(CLASS_NAMES) else str(pred_cls[i].item()),
                round(iou_val, 3),
                wandb.Image(gt_colored),
                wandb.Image(pred_colored),
            )
            count += 1
        if count >= 20:
            break

wandb.log({"pipeline_showcase": table})
wandb.finish()
print("✅  §2.7 pipeline table logged to W&B")

## 7 · Upload `unet.pth` to Google Drive

1. Kaggle right sidebar → **Output** → download `checkpoints/unet.pth`
2. [drive.google.com](https://drive.google.com) → upload → **Share: Anyone with the link**
3. Copy **FILE_ID** from share URL and paste below

In [ ]:
UNET_DRIVE_ID = "PASTE_FILE_ID_HERE"

import re
multitask_path = "models/multitask.py"
with open(multitask_path) as f:
    content = f.read()
content = re.sub(
    r'UNET_DRIVE_ID\s*=\s*"[^"]*"',
    f'UNET_DRIVE_ID       = "{UNET_DRIVE_ID}"',
    content
)
with open(multitask_path, "w") as f:
    f.write(content)
print(f"✅  Updated models/multitask.py → UNET_DRIVE_ID = '{UNET_DRIVE_ID}'")

# Sanity check
import gdown, tempfile, os
with tempfile.NamedTemporaryFile(suffix=".pth", delete=False) as tmp:
    tmp_path = tmp.name
try:
    gdown.download(id=UNET_DRIVE_ID, output=tmp_path, quiet=False, fuzzy=True)
    print(f"✅  Download test passed ({os.path.getsize(tmp_path)/1e6:.1f} MB)")
except Exception as e:
    print(f"❌  Download test FAILED: {e}")
finally:
    os.unlink(tmp_path)

## 8 · Push to GitHub → Submit to Gradescope

In [ ]:
!git config user.email "naveen@student.iitm.ac.in"
!git config user.name  "Naveen"
!git add models/multitask.py
!git diff --cached --stat
!git commit -m "task3: set UNET_DRIVE_ID for Gradescope submission"
!git push
print("\n🎉  All tasks complete! Gradescope will now test 4.3a/b/c.")